# Лабораторная работа №1: Моделирование системы массового обслуживания M/M/1/0

## Описание модели

Система представляет собой одноканальную систему массового обслуживания с отказами (M/M/1/0) — модель key-value базы данных.

**Основные параметры:**
- $\lambda$ (lambda) — интенсивность входящего потока заявок (заявок/сек), Пуассоновский поток
- $\mu$ (mu) — интенсивность обслуживания (заявок/сек), экспоненциальное время обслуживания
- $K = 0$ — размер буфера (очереди отсутствует)
- $n = 1$ — количество каналов обслуживания

**Принцип работы:**
- Если канал свободен — заявка принимается на обслуживание
- Если канал занят — заявка сразу теряется (отказ)

## Теоретические формулы

### Коэффициент загрузки:
$$\rho = \frac{\lambda}{\mu}$$

### Относительная пропускная способность:
$$Q = P_0 = \frac{\mu}{\lambda + \mu}$$

### Абсолютная пропускная способность:
$$A = \lambda \cdot Q = \frac{\lambda \mu}{\lambda + \mu}$$

### Вероятность отказа:
$$P_{\text{отк}} = P_1 = \frac{\lambda}{\lambda + \mu}$$

### Среднее время пребывания заявки в системе:
$$W = \frac{1}{\mu}$$

### Среднее количество заявок в системе:
$$L = \rho = \frac{\lambda}{\mu}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple

# Настройка графиков
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 6)

@dataclass
class SimulationResults:
    """Результаты моделирования системы M/M/1/0"""
    total_requests: int          # Общее количество поступивших заявок
    served_requests: int         # Количество обслуженных заявок
    lost_requests: int           # Количество потерянных заявок
    rejection_probability: float  # Экспериментальная вероятность отказа
    channel_busy_time: float     # Время, когда канал был занят
    channel_utilization: float   # Коэффициент загрузки канала
    simulation_time: float       # Общее время моделирования
    arrival_times: List[float]   # Времена поступления заявок
    service_times: List[float]   # Времена обслуживания заявок

## Функции теоретических расчетов

In [ ]:
def theoretical_metrics(lambda_: float, mu_: float) -> dict:
    """
    Вычисление теоретических характеристик системы M/M/1/0.
    
    Параметры:
    ----------
    lambda_ : float
        Интенсивность входящего потока заявок
    mu_ : float
        Интенсивность обслуживания
    
    Возвращает:
    -----------
    dict : словарь с теоретическими характеристиками
    """
    rho = lambda_ / mu_
    P0 = mu_ / (lambda_ + mu_)  # Вероятность, что канал свободен
    P1 = lambda_ / (lambda_ + mu_)  # Вероятность, что канал занят
    
    return {
        'rho': rho,
        'P0': P0,
        'P1': P1,
        'Q': P0,  # Относительная пропускная способность
        'A': lambda_ * P0,  # Абсолютная пропускная способность
        'P_reject': P1,  # Вероятность отказа
        'W': 1 / mu_,  # Среднее время пребывания
        'L': rho  # Среднее количество заявок
    }

## Функция имитационного моделирования

In [ ]:
def simulate_mm10(lambda_: float, mu_: float, simulation_time: float, seed: int = None) -> SimulationResults:
    """
    Имитационное моделирование системы M/M/1/0.
    
    Параметры:
    ----------
    lambda_ : float
        Интенсивность входящего потока заявок
    mu_ : float
        Интенсивность обслуживания
    simulation_time : float
        Время моделирования
    seed : int, optional
        Seed для генератора случайных чисел
    
    Возвращает:
    -----------
    SimulationResults : результаты моделирования
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Состояние системы
    channel_busy = False  # Занят ли канал
    service_end_time = 0.0  # Время завершения обслуживания
    
    # Счётчики
    total_requests = 0
    served_requests = 0
    lost_requests = 0
    channel_busy_time = 0.0
    
    # Для хранения времен
    arrival_times = []
    service_times = []
    
    # Первое поступление заявки
    next_arrival = np.random.exponential(1 / lambda_)
    
    # Имитационное время
    current_time = 0.0
    last_busy_change_time = 0.0
    
    while current_time < simulation_time:
        # Определяем следующее событие
        if channel_busy:
            # Следующее событие: завершение обслуживания
            next_event = min(next_arrival, service_end_time)
        else:
            # Канал свободен, следующее событие: поступление заявки
            next_event = next_arrival
        
        # Проверяем, не вышли ли за пределы времени моделирования
        if next_event > simulation_time:
            # Фиксируем время занятости канала до конца моделирования
            if channel_busy:
                channel_busy_time += simulation_time - last_busy_change_time
            break
        
        # Переходим к следующему событию
        time_diff = next_event - current_time
        current_time = next_event
        
        # Обработка события
        if abs(current_time - next_arrival) < 1e-10:  # Поступление заявки
            total_requests += 1
            arrival_times.append(current_time)
            
            if not channel_busy:
                # Канал свободен - принимаем заявку
                served_requests += 1
                
                # Время обслуживания
                service_time = np.random.exponential(1 / mu_)
                service_times.append(service_time)
                
                # Канал становится занятым
                channel_busy = True
                service_end_time = current_time + service_time
                last_busy_change_time = current_time
            else:
                # Канал занят - заявка теряется
                lost_requests += 1
            
            # Планируем следующее поступление
            next_arrival = current_time + np.random.exponential(1 / lambda_)
            
        elif abs(current_time - service_end_time) < 1e-10:  # Завершение обслуживания
            # Канал освобождается
            channel_busy_time += current_time - last_busy_change_time
            channel_busy = False
            last_busy_change_time = current_time
    
    return SimulationResults(
        total_requests=total_requests,
        served_requests=served_requests,
        lost_requests=lost_requests,
        rejection_probability=lost_requests / total_requests if total_requests > 0 else 0,
        channel_busy_time=channel_busy_time,
        channel_utilization=channel_busy_time / simulation_time,
        simulation_time=simulation_time,
        arrival_times=arrival_times,
        service_times=service_times
    )

## Пример 1: Базовый эксперимент

Проведем моделирование для заданных параметров и сравним с теорией.

In [ ]:
# Параметры эксперимента
lambda_exp = 5.0   # заявок/сек
mu_exp = 6.0        # заявок/сек
T_exp = 1000.0      # время моделирования

print(f"Параметры эксперимента:")
print(f"  λ (lambda) = {lambda_exp} заявок/сек")
print(f"  μ (mu) = {mu_exp} заявок/сек")
print(f"  Время моделирования = {T_exp} сек")
print()

# Теоретические значения
theor = theoretical_metrics(lambda_exp, mu_exp)
print("Теоретические значения:")
print(f"  Коэффициент загрузки (ρ) = {theor['rho']:.4f}")
print(f"  Вероятность отказа (P_отк) = {theor['P_reject']:.4f}")
print(f"  Относительная пропускная способность (Q) = {theor['Q']:.4f}")
print(f"  Абсолютная пропускная способность (A) = {theor['A']:.4f} заявок/сек")
print(f"  Среднее время пребывания (W) = {theor['W']:.4f} сек")
print(f"  Среднее количество заявок (L) = {theor['L']:.4f}")
print()

# Имитационное моделирование
sim = simulate_mm10(lambda_exp, mu_exp, T_exp, seed=42)

print("Результаты моделирования:")
print(f"  Поступило заявок: {sim.total_requests}")
print(f"  Обслужено заявок: {sim.served_requests}")
print(f"  Потеряно заявок: {sim.lost_requests}")
print(f"  Вероятность отказа (эксп.) = {sim.rejection_probability:.4f}")
print(f"  Загрузка канала (эксп.) = {sim.channel_utilization:.4f}")
print()

print("Сравнение теории и эксперимента:")
print(f"  P_отк (теория) = {theor['P_reject']:.4f}")
print(f"  P_отк (эксп.) = {sim.rejection_probability:.4f}")
print(f"  Разница = {abs(theor['P_reject'] - sim.rejection_probability):.4f} ({abs(theor['P_reject'] - sim.rejection_probability) / theor['P_reject'] * 100:.2f}%)")
print()
print(f"  ρ (теория) = {theor['rho']:.4f}")
print(f"  ρ (эксп.) = {sim.channel_utilization:.4f}")
print(f"  Разница = {abs(theor['rho'] - sim.channel_utilization):.4f}")

## Пример 2: Множество экспериментов для одного набора параметров

Проведем несколько запусков моделирования и посмотрим на разброс результатов.

In [ ]:
# Параметры
lambda_exp = 5.0
mu_exp = 6.0
T_exp = 1000.0
n_runs = 10

# Запуск нескольких экспериментов
results = []
for i in range(n_runs):
    sim = simulate_mm10(lambda_exp, mu_exp, T_exp, seed=i)
    results.append({
        'run': i + 1,
        'rejection_prob': sim.rejection_probability,
        'utilization': sim.channel_utilization,
        'total': sim.total_requests,
        'served': sim.served_requests,
        'lost': sim.lost_requests
    })

df = pd.DataFrame(results)

print("Результаты множества экспериментов:")
print(df.to_string(index=False))
print()

print("Статистика по вероятности отказа:")
print(f"  Среднее: {df['rejection_prob'].mean():.4f}")
print(f"  Стандартное отклонение: {df['rejection_prob'].std():.4f}")
print(f"  Минимум: {df['rejection_prob'].min():.4f}")
print(f"  Максимум: {df['rejection_prob'].max():.4f}")
print()

theor = theoretical_metrics(lambda_exp, mu_exp)
print(f"Теоретическое значение P_отк: {theor['P_reject']:.4f}")

## Пример 3: Зависимость вероятности отказа от λ

Построим график зависимости вероятности отказа от интенсивности входящего потока при фиксированном μ.

In [ ]:
# Параметры
mu_fixed = 6.0
T_exp = 1000.0
lambda_values = np.linspace(0.5, 15.0, 30)

# Теоретические значения
theor_p_reject = []
theor_util = []
for lam in lambda_values:
    theor = theoretical_metrics(lam, mu_fixed)
    theor_p_reject.append(theor['P_reject'])
    theor_util.append(theor['rho'])

# Экспериментальные значения
exp_p_reject = []
exp_util = []

for lam in lambda_values:
    sim = simulate_mm10(lam, mu_fixed, T_exp, seed=42)
    exp_p_reject.append(sim.rejection_probability)
    exp_util.append(sim.channel_utilization)

# Построение графиков
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# График вероятности отказа
ax1.plot(lambda_values, theor_p_reject, 'b-', linewidth=2, label='Теория')
ax1.plot(lambda_values, exp_p_reject, 'ro-', markersize=4, label='Эксперимент')
ax1.axvline(mu_fixed, color='green', linestyle='--', alpha=0.7, label=f'λ = μ ({mu_fixed})')
ax1.set_xlabel('Интенсивность входящего потока, λ (заявок/сек)')
ax1.set_ylabel('Вероятность отказа')
ax1.set_title('Зависимость вероятности отказа от λ')
ax1.legend()
ax1.grid(True, alpha=0.3)

# График загрузки канала
ax2.plot(lambda_values, theor_util, 'b-', linewidth=2, label='Теория')
ax2.plot(lambda_values, exp_util, 'ro-', markersize=4, label='Эксперимент')
ax2.axvline(mu_fixed, color='green', linestyle='--', alpha=0.7, label=f'λ = μ ({mu_fixed})')
ax2.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='ρ = 1')
ax2.set_xlabel('Интенсивность входящего потока, λ (заявок/сек)')
ax2.set_ylabel('Коэффициент загрузки, ρ')
ax2.set_title('Зависимость загрузки канала от λ')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Вывод таблицы значений
df_compare = pd.DataFrame({
    'λ': lambda_values,
    'P_отк (теория)': theor_p_reject,
    'P_отк (эксп.)': exp_p_reject,
    'Разница': [abs(t - e) for t, e in zip(theor_p_reject, exp_p_reject)]
})
print("\nСравнение теории и эксперимента:")
print(df_compare.to_string(index=False))

## Пример 4: Зависимость от μ при фиксированном λ

In [ ]:
# Параметры
lambda_fixed = 5.0
T_exp = 1000.0
mu_values = np.linspace(1.0, 15.0, 30)

# Теоретические значения
theor_p_reject = []
theor_util = []
theor_Q = []
theor_A = []

for mu in mu_values:
    theor = theoretical_metrics(lambda_fixed, mu)
    theor_p_reject.append(theor['P_reject'])
    theor_util.append(theor['rho'])
    theor_Q.append(theor['Q'])
    theor_A.append(theor['A'])

# Экспериментальные значения
exp_p_reject = []
exp_util = []

for mu in mu_values:
    sim = simulate_mm10(lambda_fixed, mu, T_exp, seed=42)
    exp_p_reject.append(sim.rejection_probability)
    exp_util.append(sim.channel_utilization)

# Построение графиков
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# График вероятности отказа
ax1.plot(mu_values, theor_p_reject, 'b-', linewidth=2, label='Теория')
ax1.plot(mu_values, exp_p_reject, 'ro-', markersize=4, label='Эксперимент')
ax1.axvline(lambda_fixed, color='green', linestyle='--', alpha=0.7, label=f'μ = λ ({lambda_fixed})')
ax1.set_xlabel('Интенсивность обслуживания, μ (заявок/сек)')
ax1.set_ylabel('Вероятность отказа')
ax1.set_title('Зависимость вероятности отказа от μ')
ax1.legend()
ax1.grid(True, alpha=0.3)

# График загрузки канала
ax2.plot(mu_values, theor_util, 'b-', linewidth=2, label='Теория')
ax2.plot(mu_values, exp_util, 'ro-', markersize=4, label='Эксперимент')
ax2.axvline(lambda_fixed, color='green', linestyle='--', alpha=0.7, label=f'μ = λ ({lambda_fixed})')
ax2.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='ρ = 1')
ax2.set_xlabel('Интенсивность обслуживания, μ (заявок/сек)')
ax2.set_ylabel('Коэффициент загрузки, ρ')
ax2.set_title('Зависимость загрузки канала от μ')
ax2.legend()
ax2.grid(True, alpha=0.3)

# График относительной пропускной способности
ax3.plot(mu_values, theor_Q, 'b-', linewidth=2, label='Теория')
ax3.axvline(lambda_fixed, color='green', linestyle='--', alpha=0.7, label=f'μ = λ ({lambda_fixed})')
ax3.set_xlabel('Интенсивность обслуживания, μ (заявок/сек)')
ax3.set_ylabel('Относительная пропускная способность, Q')
ax3.set_title('Зависимость относительной пропускной способности от μ')
ax3.legend()
ax3.grid(True, alpha=0.3)

# График абсолютной пропускной способности
ax4.plot(mu_values, theor_A, 'b-', linewidth=2, label='Теория')
ax4.axvline(lambda_fixed, color='green', linestyle='--', alpha=0.7, label=f'μ = λ ({lambda_fixed})')
ax4.set_xlabel('Интенсивность обслуживания, μ (заявок/сек)')
ax4.set_ylabel('Абсолютная пропускная способность, A (заявок/сек)')
ax4.set_title('Зависимость абсолютной пропускной способности от μ')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Пример 5: Исследование сходимости эксперимента

Проверим, как увеличивается точность моделирования с ростом времени симуляции.

In [ ]:
# Параметры
lambda_exp = 5.0
mu_exp = 6.0
time_values = np.logspace(1, 4, 20)  # от 10 до 10000

theor = theoretical_metrics(lambda_exp, mu_exp)
theor_p_reject = theor['P_reject']

exp_p_reject = []
exp_util = []

for T in time_values:
    sim = simulate_mm10(lambda_exp, mu_exp, T, seed=42)
    exp_p_reject.append(sim.rejection_probability)
    exp_util.append(sim.channel_utilization)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# График сходимости по вероятности отказа
ax1.semilogx(time_values, exp_p_reject, 'bo-', markersize=4, label='Эксперимент')
ax1.axhline(theor_p_reject, color='red', linestyle='--', linewidth=2, label=f'Теория ({theor_p_reject:.4f})')
ax1.set_xlabel('Время моделирования (сек)')
ax1.set_ylabel('Вероятность отказа')
ax1.set_title('Сходимость эксперимента по вероятности отказа')
ax1.legend()
ax1.grid(True, alpha=0.3)

# График отклонения от теории
errors = [abs(e - theor_p_reject) for e in exp_p_reject]
ax2.semilogx(time_values, errors, 'bo-', markersize=4)
ax2.set_xlabel('Время моделирования (сек)')
ax2.set_ylabel('Абсолютная ошибка')
ax2.set_title('Зависимость ошибки от времени моделирования')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Теоретическое значение P_отк: {theor_p_reject:.6f}")
print(f"\nЗначения P_отк при разных временах моделирования:")
for T, p in zip(time_values, exp_p_reject):
    print(f"  T = {T:8.1f} сек: P_отк = {p:.6f}, ошибка = {abs(p - theor_p_reject):.6f}")

## Пример 6: Визуализация процесса моделирования

Покажем временные моменты поступления и обслуживания заявок.

In [ ]:
# Параметры для визуализации
lambda_exp = 5.0
mu_exp = 6.0
T_exp = 50.0

sim = simulate_mm10(lambda_exp, mu_exp, T_exp, seed=123)

# Построение временной диаграммы
fig, ax = plt.subplots(figsize=(14, 4))

# Рисуем поступления
for i, arr_time in enumerate(sim.arrival_times[:30]):  # Первые 30 заявок
    if i < 30:
        ax.plot(arr_time, 1, 'rv', markersize=8, label='Поступление' if i == 0 else "")
        ax.text(arr_time, 1.1, str(i+1), fontsize=8, ha='center')

# Рисуем интервалы обслуживания
service_start = 0
for i, (arr_time, service_time) in enumerate(zip(sim.arrival_times[:30], sim.service_times[:30])):
    if service_start <= arr_time:
        service_start = arr_time
    else:
        service_start = service_start
    
    service_end = service_start + service_time
    ax.plot([service_start, service_end], [0.5, 0.5], 'b-', linewidth=3)
    service_start = service_end

# Рисуем состояние канала
channel_busy_time = []
for i, (arr_time, service_time) in enumerate(zip(sim.arrival_times, sim.service_times)):
    if service_time > 0:
        channel_busy_time.append((arr_time, arr_time + service_time))

ax.set_xlabel('Время (сек)')
ax.set_yticks([])
ax.set_title(f'Временная диаграмма работы системы (λ={lambda_exp}, μ={mu_exp})')
ax.legend(['Канал занят', 'Поступление заявки'])
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.5)
ax.set_xlim(0, min(T_exp, 30))

plt.tight_layout()
plt.show()

print(f"Всего поступило: {sim.total_requests} заявок")
print(f"Обслужено: {sim.served_requests} заявок")
print(f"Потеряно: {sim.lost_requests} заявок")
print(f"Вероятность отказа: {sim.rejection_probability:.4f}")

## Пример 7: Исследование критической области (λ → μ)

Исследуем поведение системы при λ, близком к μ.

In [ ]:
# Параметры
mu_fixed = 10.0
T_exp = 2000.0

# Значения λ вблизи μ
lambda_values = np.linspace(mu_fixed * 0.5, mu_fixed * 1.5, 15)

theor_p_reject = []
exp_p_reject = []
exp_p_reject_std = []

for lam in lambda_values:
    theor = theoretical_metrics(lam, mu_fixed)
    theor_p_reject.append(theor['P_reject'])
    
    # Множество экспериментов для оценки разброса
    exp_values = []
    for seed in range(10):
        sim = simulate_mm10(lam, mu_fixed, T_exp, seed=seed)
        exp_values.append(sim.rejection_probability)
    
    exp_p_reject.append(np.mean(exp_values))
    exp_p_reject_std.append(np.std(exp_values))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(lambda_values, theor_p_reject, 'b-', linewidth=2, label='Теория')
ax.plot(lambda_values, exp_p_reject, 'ro-', markersize=6, label='Эксперимент (среднее)')
ax.axvline(mu_fixed, color='green', linestyle='--', alpha=0.7, label=f'λ = μ ({mu_fixed})')
ax.axvline(mu_fixed * 0.5, color='orange', linestyle=':', alpha=0.5, label=f'λ = μ/2 ({mu_fixed * 0.5})')
ax.axvline(mu_fixed * 2, color='orange', linestyle=':', alpha=0.5, label=f'λ = 2μ ({mu_fixed * 2})')

# Отображаем стандартное отклонение как заштрихованную область
ax.fill_between(lambda_values, 
                [m - s for m, s in zip(exp_p_reject, exp_p_reject_std)],
                [m + s for m, s in zip(exp_p_reject, exp_p_reject_std)],
                alpha=0.2, color='red', label='±1 стандартное отклонение')

ax.set_xlabel('Интенсивность входящего потока, λ (заявок/сек)')
ax.set_ylabel('Вероятность отказа')
ax.set_title(f'Зависимость вероятности отказа от λ (μ = {mu_fixed})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Анализ критической области:")
print(f"При λ = μ/2 = {mu_fixed * 0.5}:")
print(f"  P_отк (теория) = {theoretical_metrics(mu_fixed * 0.5, mu_fixed)['P_reject']:.4f}")
print(f"  P_отк (эксп.) = {exp_p_reject[int(np.where(lambda_values == mu_fixed * 0.5)[0][0])]:.4f}")
print()
print(f"При λ = μ = {mu_fixed}:")
print(f"  P_отк (теория) = {theoretical_metrics(mu_fixed, mu_fixed)['P_reject']:.4f}")
print(f"  P_отк (эксп.) = {exp_p_reject[int(np.where(lambda_values == mu_fixed)[0][0])]:.4f}")
print()
print(f"При λ = 2μ = {mu_fixed * 2}:")
print(f"  P_отк (теория) = {theoretical_metrics(mu_fixed * 2, mu_fixed)['P_reject']:.4f}")
print(f"  P_отк (эксп.) = {exp_p_reject[int(np.where(lambda_values == mu_fixed * 2)[0][0])]:.4f}")

## Пример 8: Интерактивный эксперимент

Измените параметры и запустите ячейку, чтобы провести свой эксперимент.

In [ ]:
# ========== НАСТРОЙКИ ЭКСПЕРИМЕНТА ==========
lambda_exp = 8.0    # Интенсивность входящего потока (заявок/сек)
mu_exp = 10.0        # Интенсивность обслуживания (заявок/сек)
T_exp = 2000.0       # Время моделирования (сек)
n_runs = 5           # Количество запусков моделирования
# ============================================

print(f"{'='*60}")
print(f"Параметры эксперимента:")
print(f"  λ = {lambda_exp} заявок/сек")
print(f"  μ = {mu_exp} заявок/сек")
print(f"  Время моделирования = {T_exp} сек")
print(f"  Количество запусков = {n_runs}")
print(f"{'='*60}\n")

# Теоретические значения
theor = theoretical_metrics(lambda_exp, mu_exp)
print("ТЕОРЕТИЧЕСКИЕ ЗНАЧЕНИЯ:")
print(f"  Коэффициент загрузки (ρ) = {theor['rho']:.4f}")
print(f"  Вероятность отказа (P_отк) = {theor['P_reject']:.4f}")
print(f"  Относительная пропускная способность (Q) = {theor['Q']:.4f}")
print(f"  Абсолютная пропускная способность (A) = {theor['A']:.4f} заявок/сек")
print(f"  Среднее время пребывания (W) = {theor['W']:.4f} сек")
print(f"  Среднее количество заявок (L) = {theor['L']:.4f}\n")

# Экспериментальные значения
exp_p_rejects = []
exp_utils = []
exp_served = []
exp_lost = []

for i in range(n_runs):
    sim = simulate_mm10(lambda_exp, mu_exp, T_exp, seed=i)
    exp_p_rejects.append(sim.rejection_probability)
    exp_utils.append(sim.channel_utilization)
    exp_served.append(sim.served_requests)
    exp_lost.append(sim.lost_requests)

print(f"РЕЗУЛЬТАТЫ МОДЕЛИРОВАНИЯ ({n_runs} запусков):")
print(f"{'Запуск':>6} | {'Всего':>6} | {'Обслужено':>9} | {'Потеряно':>9} | {'P_отк':>6} | {'ρ':>6}")
print("-" * 60)
for i in range(n_runs):
    sim = simulate_mm10(lambda_exp, mu_exp, T_exp, seed=i)
    print(f"{i+1:>6} | {sim.total_requests:>6} | {sim.served_requests:>9} | {sim.lost_requests:>9} | {sim.rejection_probability:>6.4f} | {sim.channel_utilization:>6.4f}")

print("-" * 60)
print(f"{'Ср.':>6} |     | {np.mean(exp_served):>9.0f} | {np.mean(exp_lost):>9.0f} | {np.mean(exp_p_rejects):>6.4f} | {np.mean(exp_utils):>6.4f}")
print(f"{'Std':>6} |     | {np.std(exp_served):>9.2f} | {np.std(exp_lost):>9.2f} | {np.std(exp_p_rejects):>6.4f} | {np.std(exp_utils):>6.4f}")
print()

# Сравнение
print("СРАВНЕНИЕ ТЕОРИИ И ЭКСПЕРИМЕНТА:")
print(f"  P_отк (теория) = {theor['P_reject']:.4f}")
print(f"  P_отк (эксп., среднее) = {np.mean(exp_p_rejects):.4f}")
print(f"  Разница = {abs(theor['P_reject'] - np.mean(exp_p_rejects)):.4f} ({abs(theor['P_reject'] - np.mean(exp_p_rejects)) / theor['P_reject'] * 100:.2f}%)")
print()
print(f"  ρ (теория) = {theor['rho']:.4f}")
print(f"  ρ (эксп., среднее) = {np.mean(exp_utils):.4f}")
print(f"  Разница = {abs(theor['rho'] - np.mean(exp_utils)):.4f}")

## Выводы

Проведенные эксперименты показывают:

1. **Сходимость к теории:** При увеличении времени моделирования экспериментальные значения сходятся к теоретическим.

2. **Влияние λ на вероятность отказа:** Вероятность отказа монотонно возрастает с ростом интенсивности входящего потока. При λ → ∞, P_отк → 1.

3. **Влияние μ на пропускную способность:** Увеличение интенсивности обслуживания снижает вероятность отказа и повышает пропускную способность.

4. **Критическая область λ ≈ μ:** При λ = μ вероятность отказа составляет 0.5. Это точка, где система работает с максимальной загрузкой.

5. **Влияние времени моделирования:** Для получения точных результатов необходимо достаточное время моделирования (обычно несколько тысяч единиц времени).